# CAWOT-CM coreset sweep — Kaggle (V2 full plan)

**Sweep config (Hướng 2.5)**:
- 4 methods: `random` / `v0_proto` / `v1` / **`v2`** (v0 farthest dropped — cited from prev run as negative control)
- 6 budgets {5, 10, 20, 30, 40, 50}% (plan Part B2 scaling curve)
- 3 seeds (42, 1, 2) for error bars
- Per-category eval: goal / full / wentwrong (wentwrong = anomaly)

Total: 4 × 6 × 3 = **72 fine-tune runs ≈ 16 h** across 3 Kaggle sessions. **Resumable** via `records.csv`.

**Setup**: GPU **P100** + Internet **ON**.

**Execution order** (this notebook):
1. Env + clone + install
2. Download train (HF, 5 shards) + Q_proxy (Drive)
3. **SMOKE TEST** V2 (~25 min) — verify V2 pipeline end-to-end before committing 16h
4. **DIAGNOSTIC** (~5 min) — image grid + PCA + Q_proxy quality (paper figures)
5. **(if multi-session) Restore records.csv** from previous session output
6. **FULL SWEEP** — 72 runs, resumable across sessions

## 1. Env + clone + install

In [ ]:
!nvidia-smi -L
import os
if not os.path.exists("/kaggle/working/cawot-cm"):
    !git clone https://github.com/HohoHocCode/cawot-cm.git /kaggle/working/cawot-cm
%cd /kaggle/working/cawot-cm
!pip install -q open_clip_torch faiss-gpu-cu12 einops huggingface_hub gdown scikit-learn

## 2a. Download train shards from HuggingFace (~7 GB / 14 GB extracted)

Skip if already cached from a previous session.

In [ ]:
!python scripts/setup_data.py --output /kaggle/working/pab_data --num-shards 5

## 2b. Download Q_proxy queries.json from Drive (~256 KB)

In [ ]:
!python scripts/setup_qproxy.py --output /kaggle/working/qproxy --only-queries

## 3. SMOKE TEST V2 (~25 min)

Verifies V2 end-to-end (1 budget × 1 seed × 1 epoch, V2 only). Runs:
- Extract image+text embeddings 50K (~18 min, **CACHED**, paid once)
- Encode 2,593 Q_proxy captions (~10 sec)
- Cluster (k=150) + V2 selection + train 1 epoch + eval

If this fails, fix before launching the 16h full sweep. If it succeeds, embeddings + Q_proxy are cached for diagnostic + full sweep (no re-extraction).

In [ ]:
!python scripts/run_sweep.py --config configs/smoke.yaml

In [ ]:
# Inspect smoke result
import json
with open("/kaggle/working/outputs_smoke/eval/summary.json") as f:
    smoke = json.load(f)
print("=== SMOKE TEST RESULT ===")
print(json.dumps(smoke, indent=2))
# Sanity: V2 at 5%/1 epoch should beat zero-shot. If lower, something's wrong.
v2_r1 = smoke.get("v2", {}).get("0.05", {}).get("overall", {}).get("mean_R@1_mean", 0)
zs_r1 = smoke.get("zeroshot", {}).get("overall", 0)
print(f"\nV2@5% mean_R@1 = {v2_r1:.2f}  vs  zero-shot = {zs_r1:.2f}")
assert v2_r1 > zs_r1, "V2 should beat zero-shot — investigate before full sweep!"

## 4. DIAGNOSTIC (~5 min) — paper figures

After smoke: embeddings + Q_proxy cached. Run diagnostic to produce:
- `selection_pca.png` — 2D PCA of train pool, each method's selection colored
- `centroid_dist_hist.png` — distance-to-centroid distribution per method
- **`image_grid_v0_vs_proto.png`** — 16 V0 (atypical) vs 16 V0_proto (prototype) images side-by-side. Visual evidence for the paper's prototype-vs-outlier narrative.
- `qproxy_quality.png` — Q_proxy pairwise sim + PCA effective dimensionality
- `qproxy_themes.txt` — 20 KMeans themes with 3 example queries each. Defends Q_proxy quality against reviewer skepticism.

In [ ]:
!python scripts/diagnose_selection.py --config config.yaml

In [ ]:
# Display the key figure inline
from IPython.display import Image
Image("/kaggle/working/cawot-cm/outputs/diagnostic/image_grid_v0_vs_proto.png")

## 5. (Multi-session only) Restore previous session's records.csv

Kaggle resets `/kaggle/working` between sessions. To resume the sweep across sessions:

**Session 1**: skip this cell — start fresh.  
**Session 2/3**: 
1. After session 1, click **Save Version** → **Save & Run All**. The `outputs/` folder becomes a notebook output.
2. In a NEW session: **Add Data** (right sidebar) → **Your Datasets** → select the previous notebook's output.
3. It mounts at `/kaggle/input/<your-notebook-name>/`. Run the cell below to copy `records.csv` back into the working directory so the sweep resumes.

Edit the `PREV_OUTPUT` path to match the mount point Kaggle gave you.

In [ ]:
# UNCOMMENT and edit PREV_OUTPUT only on session 2+
# import shutil, os
# PREV_OUTPUT = "/kaggle/input/cawot-cm-session-1/outputs"   # ← edit to your prev session's mount
# dst = "/kaggle/working/outputs"
# os.makedirs(f"{dst}/eval", exist_ok=True)
# os.makedirs(f"{dst}/embeddings", exist_ok=True)
# for sub in ("eval/records.csv", "eval/summary.json",
#             "embeddings/image_embeddings.npy",
#             "embeddings/text_embeddings.npy",
#             "embeddings/qproxy_clip_text_emb.npy"):
#     src = f"{PREV_OUTPUT}/{sub}"
#     if os.path.exists(src):
#         shutil.copy(src, f"{dst}/{sub}")
#         print(f"restored {sub}")
#     else:
#         print(f"(skip) {src} not found")

## 6. FULL SWEEP — 72 runs, resumable

Reads `records.csv` at start, skips any (method, budget, seed) already done. Default config:
- methods: random, v0_proto, v1, v2
- budgets: 5/10/20/30/40/50%
- seeds: 42, 1, 2

**Session budget** (~5.5 h each):
- Session 1: seeds 42 done → roughly 24 of 72 runs
- Session 2: seed 1 → roughly 24 more
- Session 3: seed 2 + cleanup → 24 more

In [ ]:
!python scripts/run_sweep.py --config config.yaml

## 7. Results table (overall + per-category)

In [ ]:
import json, pandas as pd
summary = json.load(open("/kaggle/working/outputs/eval/summary.json"))
print("zeroshot:", summary["zeroshot"])
methods = [m for m in ["random", "v0_proto", "v1", "v2"] if m in summary]
budgets = sorted({float(b) for m in methods for b in summary[m].keys()})
categories = ["overall", "goal", "full", "wentwrong"]
rows = []
for c in categories:
    for b in budgets:
        r = {"category": c, "budget": f"{int(b*100)}%"}
        for m in methods:
            s = summary[m].get(str(b), {}).get(c)
            r[m] = f"{s['mean_R@1_mean']:.2f}±{s['mean_R@1_std']:.2f}" if s else "-"
        rows.append(r)
pd.DataFrame(rows)

## 8. Plot R@1 vs budget — overall + wentwrong (anomaly)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
markers = {"random": "o", "v0_proto": "v", "v1": "s", "v2": "D"}
for ax, cat in zip(axes, ["overall", "wentwrong"]):
    x = [b * 100 for b in budgets]
    for m in methods:
        y = [summary[m].get(str(b), {}).get(cat, {}).get("mean_R@1_mean") for b in budgets]
        e = [summary[m].get(str(b), {}).get(cat, {}).get("mean_R@1_std", 0) for b in budgets]
        if all(v is None for v in y): continue
        ax.errorbar(x, y, yerr=e, marker=markers.get(m, "o"), capsize=3, label=m)
    zs = summary["zeroshot"].get(cat)
    if zs is not None:
        ax.axhline(zs, ls="--", c="gray", label=f"zero-shot ({zs:.1f})")
    ax.set_xlabel("Budget (% of train pool)"); ax.set_ylabel("mean R@1")
    ax.set_title(cat); ax.legend(); ax.grid(alpha=0.3)
fig.suptitle("V2 sweep — Overall vs. Anomaly (wentwrong)")
plt.tight_layout()
plt.savefig("/kaggle/working/outputs/eval/sweep_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Artifacts for the paper

Save Version → all outputs committed:
- `outputs/eval/records.csv` — 72 runs × 4 categories = 288 rows + zero-shot
- `outputs/eval/summary.json` — aggregated mean ± std
- `outputs/eval/sweep_curve.png` — main figure (overall + wentwrong)
- `outputs/diagnostic/*.png + qproxy_themes.txt` — diagnostic figures + theme report